# Error de medición: observa cómo se encoge la pendiente

Complemento de las páginas del curso *Error de medición* y *Cómo: corregir el error de medición*.

Todo lo que ves aquí está simulado a partir de una verdad conocida, así que puedes ver exactamente qué hace el error de medición:

1. Un **predictor** ruidoso sesga la pendiente hacia cero, y **tener más datos no lo arregla**.
2. El ruido en la **variable de respuesta** no sesga la pendiente; solo la hace menos precisa.
3. Con **mediciones repetidas** puedes estimar la fiabilidad y corregir el sesgo (a costa de una estimación más ruidosa).

Cambia los parámetros de la siguiente celda y usa *Run All* para ver cómo cambia el panorama. Necesita `numpy`, `pandas` y `matplotlib`.

In [ ]:
# ---- Parameters: change these and re-run ----
TRUE_SLOPE = 0.5      # true effect of x on y (e.g. mmHg of outcome per mmHg of blood pressure)
SD_TRUE_X = 12        # spread of the true predictor values across units
SD_ERROR_X = 8        # measurement-error SD of ONE reading of x (0 = perfect measurement)
SD_ERROR_Y = 0        # measurement-error SD added to the outcome (0 = none)
SD_NOISE_Y = 8        # ordinary scatter of y around the true line
SAMPLE_SIZES = [25, 100, 400, 1600, 6400]   # sample sizes for experiment 1
N_CORRECTION = 400    # sample size for experiments 2 and 3
N_SIMS = 300          # simulated datasets per setting
SEED = 0

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(SEED)


def simulate(n, sd_error_x, sd_error_y):
    """Two independent readings of x (x1, x2) and the outcome y; the true x is never observed."""
    x_true = rng.normal(0, SD_TRUE_X, n)
    y = TRUE_SLOPE * x_true + rng.normal(0, SD_NOISE_Y, n) + rng.normal(0, sd_error_y, n)
    x1 = x_true + rng.normal(0, sd_error_x, n)
    x2 = x_true + rng.normal(0, sd_error_x, n)
    return x1, x2, y


def slope(x, y):
    return np.cov(x, y)[0, 1] / x.var(ddof=1)


reliability = SD_TRUE_X**2 / (SD_TRUE_X**2 + SD_ERROR_X**2)
print(f"Reliability of one reading: {reliability:.2f}")
print(f"Expected slope with the noisy predictor, at ANY sample size: {TRUE_SLOPE * reliability:.2f} (true slope {TRUE_SLOPE})")

## 1. Un predictor ruidoso: el sesgo no disminuye con el tamaño muestral

Para cada tamaño muestral simulamos muchos conjuntos de datos y ajustamos la pendiente de la regresión ordinaria con una única lectura ruidosa. La banda muestra dónde cae el 95 % de las estimaciones. Las muestras mayores estrechan la banda, pero se estrecha alrededor del valor *atenuado*, no del verdadero.

In [ ]:
rows = []
for n in SAMPLE_SIZES:
    est = []
    for _ in range(N_SIMS):
        x1, _, y = simulate(n, SD_ERROR_X, SD_ERROR_Y)
        est.append(slope(x1, y))
    rows.append({"n": n, "mean": np.mean(est), "low": np.percentile(est, 2.5), "high": np.percentile(est, 97.5)})
exp1 = pd.DataFrame(rows).set_index("n")
print(exp1.round(3))

fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(exp1.index, exp1["low"], exp1["high"], alpha=0.25, label="95% of estimates")
ax.plot(exp1.index, exp1["mean"], marker="o", label="mean estimate (naive slope)")
ax.axhline(TRUE_SLOPE, color="green", linestyle="--", label="true slope")
ax.axhline(TRUE_SLOPE * reliability, color="red", linestyle=":", label="true slope x reliability")
ax.set_xscale("log")
ax.set_xlabel("sample size")
ax.set_ylabel("estimated slope")
ax.legend()
plt.show()

## 2. Ruido en el predictor vs. ruido en la variable de respuesta

Mismo tamaño muestral, cuatro escenarios. Fíjate en la columna *mean* (el sesgo) y en la columna *SD* (cuánto se mueve la estimación de un conjunto de datos a otro).

In [ ]:
# The outcome-error scenarios use a visible amount of noise even if SD_ERROR_Y is 0 above.
noisy_y = SD_ERROR_Y if SD_ERROR_Y > 0 else 20
scenarios = {
    "no measurement error": (0, 0),
    "error in x only": (SD_ERROR_X, 0),
    "error in y only": (0, noisy_y),
    "error in both": (SD_ERROR_X, noisy_y),
}
rows = []
for name, (sd_x, sd_y) in scenarios.items():
    est = []
    for _ in range(N_SIMS):
        x1, _, y = simulate(N_CORRECTION, sd_x, sd_y)
        est.append(slope(x1, y))
    rows.append({"scenario": name, "mean slope": np.mean(est), "SD of slope": np.std(est)})
print(f"True slope: {TRUE_SLOPE}")
pd.DataFrame(rows).set_index("scenario").round(3)

## 3. Corregir la pendiente con mediciones repetidas

Con dos lecturas por unidad las promediamos, estimamos la varianza del error de medición a partir de cuánto discrepan las dos lecturas, calculamos la fiabilidad y dividimos la pendiente ingenua entre ella. Es la receta de la página práctica.

In [ ]:
def naive_and_corrected(x1, x2, y):
    x = (x1 + x2) / 2
    var_error = ((x1 - x2) ** 2).mean() / 2
    rel = 1 - (var_error / 2) / x.var(ddof=1)
    naive = slope(x, y)
    return naive, naive / rel


results = np.array([naive_and_corrected(*simulate(N_CORRECTION, SD_ERROR_X, SD_ERROR_Y)) for _ in range(N_SIMS)])
naive_est, corrected_est = results[:, 0], results[:, 1]
print(f"true slope {TRUE_SLOPE}")
print(f"naive:     mean {naive_est.mean():.3f}, SD {naive_est.std():.3f}")
print(f"corrected: mean {corrected_est.mean():.3f}, SD {corrected_est.std():.3f}")

fig, ax = plt.subplots(figsize=(7, 4))
bins = np.linspace(min(naive_est.min(), corrected_est.min()), max(naive_est.max(), corrected_est.max()), 40)
ax.hist(naive_est, bins=bins, alpha=0.6, label="naive (averaged replicates)")
ax.hist(corrected_est, bins=bins, alpha=0.6, label="corrected")
ax.axvline(TRUE_SLOPE, color="green", linestyle="--", label="true slope")
ax.set_xlabel("estimated slope")
ax.set_ylabel("number of simulated datasets")
ax.legend()
plt.show()

## Cosas para probar

- Pon `SD_ERROR_X = 0`: las líneas roja y verde del experimento 1 coinciden y el sesgo desaparece.
- Sube `SD_ERROR_X` a 20: la fiabilidad baja y la pendiente ingenua se hace mucho menor, con cualquier tamaño muestral.
- Pon `SAMPLE_SIZES = [10, 20, 40]` y `N_CORRECTION = 30`: la estimación corregida ya no está sesgada, pero es mucho más ruidosa que la ingenua. La corrección cambia sesgo por varianza, así que necesita una muestra razonable.
- Pon `TRUE_SLOPE = 0`: el predictor ruidoso no crea un efecto de la nada; la atenuación solo encoge los efectos reales.
- Pon `SD_ERROR_Y = 30` y compara los experimentos 1 y 2: el error en la variable de respuesta ensancha la banda sin mover su centro.

Ten en cuenta que la corrección del experimento 3 solo es válida para un único predictor ruidoso con error puramente aleatorio; consulta los límites que se enumeran en la página práctica.